###Bronze Data Ingestion

In [0]:
dbutils.widgets.text(
    "source_path",
    "s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv"
)
dbutils.widgets.text("catalog", "marketing_catalog")
dbutils.widgets.text("schema", "marketing_bronze")
dbutils.widgets.text("table_name", "campaigns")

source_path = dbutils.widgets.get("source_path").strip()
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
table_name = dbutils.widgets.get("table_name").strip()

# Validate identifiers before using them in SQL.
import re

for name in [catalog, schema, table_name]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid catalog/schema/table name: {name}")

target_table = f"{catalog}.{schema}.{table_name}"

print("Source:", source_path)
print("Target:", target_table)

Source: s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv
Target: marketing_catalog.marketing_bronze.campaigns


###Cell 2 — Read raw data and add audit columns

Keep all source fields as strings. Date, currency, and duration conversion belong in Silver.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

batch_id = str(uuid.uuid4())
ingested_at = datetime.now(timezone.utc)

source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(source_path)
)

bronze_df = (
    source_df
    .withColumn("_source_file", F.lit(source_path))
    .withColumn("_ingest_batch_id", F.lit(batch_id))
    .withColumn(
        "_ingested_at_utc",
        F.lit(ingested_at).cast("timestamp")
    )
)

print("Source columns:", source_df.columns)
display(bronze_df.limit(10))

Source columns: ['Campaign_ID', 'Company', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Location', 'Language', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment', 'Date']


Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,_source_file,_ingest_batch_id,_ingested_at_utc
1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.5,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
6,DataTech Solutions,Display,All Ages,15 days,Instagram,0.07,"$9,716.00",4.36,New York,German,100,1643,1,Foodies,2021-01-06,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
7,NexGen Systems,Email,Women 35-44,60 days,Website,0.13,"$11,067.00",2.86,Los Angeles,Spanish,817,8749,10,Tech Enthusiasts,2021-01-07,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
8,DataTech Solutions,Search,Men 18-24,45 days,Google Ads,0.08,"$13,280.00",5.55,Los Angeles,Mandarin,624,7854,7,Outdoor Adventurers,2021-01-08,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
9,Alpha Innovations,Social Media,Women 35-44,15 days,Facebook,0.09,"$18,066.00",6.73,Chicago,German,861,1754,6,Tech Enthusiasts,2021-01-09,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
10,TechCorp,Email,Women 35-44,15 days,Instagram,0.09,"$13,766.00",3.78,Los Angeles,English,642,3856,3,Tech Enthusiasts,2021-01-10,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z


FAILFAST stops ingestion if Spark encounters malformed CSV records, so they aren’t silently skipped.


###Cell 3 — Validate before writing

In [0]:
expected_columns = {
    "Campaign_ID", "Company", "Campaign_Type", "Target_Audience",
    "Duration", "Channel_Used", "Conversion_Rate",
    "Acquisition_Cost", "ROI", "Location", "Language",
    "Clicks", "Impressions", "Engagement_Score",
    "Customer_Segment", "Date"
}

actual_columns = set(source_df.columns)

if actual_columns != expected_columns:
    raise ValueError(
        f"Source schema changed. "
        f"Missing: {sorted(expected_columns - actual_columns)}; "
        f"Unexpected: {sorted(actual_columns - expected_columns)}"
    )

source_count = source_df.count()

if source_count == 0:
    raise ValueError("Source is empty. Bronze table will not be overwritten.")

print("Source rows:", source_count)
print("Source columns:", len(source_df.columns))
print("Batch ID:", batch_id)

Source rows: 200000
Source columns: 16
Batch ID: d1199657-66e2-4c82-b5b9-ebc4248e95d1


###Cell 4 — Save the Bronze snapshot

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

print("Saved:", target_table)

Saved: marketing_catalog.marketing_bronze.campaigns


This replaces the current snapshot when rerun, preventing the doubled rows we encountered earlier. It preserves every source row, including duplicates already present in the CSV. It does not retain previous ingestion snapshots

###Cell 5 — Verify the saved table

In [0]:
saved_df = spark.table(target_table)
saved_count = saved_df.count()

if saved_count != source_count:
    raise RuntimeError(
        f"Row count mismatch: source={source_count}, saved={saved_count}"
    )

print("Validation: PASSED")
print("Bronze rows:", saved_count)
print("Bronze columns:", len(saved_df.columns))

display(saved_df.limit(10))

Validation: PASSED
Bronze rows: 200000
Bronze columns: 19


Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,_source_file,_ingest_batch_id,_ingested_at_utc
1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.5,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
6,DataTech Solutions,Display,All Ages,15 days,Instagram,0.07,"$9,716.00",4.36,New York,German,100,1643,1,Foodies,2021-01-06,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
7,NexGen Systems,Email,Women 35-44,60 days,Website,0.13,"$11,067.00",2.86,Los Angeles,Spanish,817,8749,10,Tech Enthusiasts,2021-01-07,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
8,DataTech Solutions,Search,Men 18-24,45 days,Google Ads,0.08,"$13,280.00",5.55,Los Angeles,Mandarin,624,7854,7,Outdoor Adventurers,2021-01-08,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
9,Alpha Innovations,Social Media,Women 35-44,15 days,Facebook,0.09,"$18,066.00",6.73,Chicago,German,861,1754,6,Tech Enthusiasts,2021-01-09,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
10,TechCorp,Email,Women 35-44,15 days,Instagram,0.09,"$13,766.00",3.78,Los Angeles,English,642,3856,3,Tech Enthusiasts,2021-01-10,s3://marketing-project-rag-ml/raw/marketing_campaign_dataset.csv,d1199657-66e2-4c82-b5b9-ebc4248e95d1,2026-10-03T13:43:24.231Z
